# Data Mining Assignment

**Aim:** To create Bronze and Silver Layers for a Sales Dataset

**Name:** Samayita Mohanta

 **Roll No.:** MSD25004

In this notebook the provided `sales.csv` file is stored unchanged in a **Bronze** layer, inspected with PySpark, given some basic preprocessing, and then the processed data is stored in **Silver** layer.

## 1. Data Lake Structure

### 1.1 Setup and dataset upload

PySpark is installed first. The notebook then looks for `sales.csv` in the Colab working directory. If the file is not there, an upload dialog opens.

In [1]:
!pip install -q pyspark

In [2]:
import os
import shutil
import hashlib

BASE_DIR = "/content" if os.path.isdir("/content") else os.getcwd()
SOURCE_FILE = os.path.join(BASE_DIR, "sales.csv")

if not os.path.isfile(SOURCE_FILE):
    try:
        from google.colab import files
        uploaded = files.upload()
        if "sales.csv" not in uploaded:
            raise FileNotFoundError("Please upload a file named exactly 'sales.csv'.")
    except ImportError:
        raise FileNotFoundError(f"sales.csv not found in {BASE_DIR}")

print("Source file:", SOURCE_FILE)
print("Size (bytes):", os.path.getsize(SOURCE_FILE))

Saving sales.csv to sales.csv
Source file: /content/sales.csv
Size (bytes): 109537


### 1.2 Creating the Bronze and Silver folders

The folder layout follows the structure given in the assignment. The raw file is copied into `bronze/`, and an MD5 checksum is compared to confirm the Bronze copy is identical to the original.

In [3]:
DATA_LAKE = os.path.join(BASE_DIR, "data_lake")
BRONZE_DIR = os.path.join(DATA_LAKE, "bronze")
SILVER_DIR = os.path.join(DATA_LAKE, "silver")
BRONZE_FILE = os.path.join(BRONZE_DIR, "sales.csv")
SILVER_PATH = os.path.join(SILVER_DIR, "preprocessed_sales")

os.makedirs(BRONZE_DIR, exist_ok=True)
os.makedirs(SILVER_DIR, exist_ok=True)

shutil.copy2(SOURCE_FILE, BRONZE_FILE)


def md5(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()


src_hash, bronze_hash = md5(SOURCE_FILE), md5(BRONZE_FILE)
assert src_hash == bronze_hash, "Bronze file does not match the original!"
print("Original MD5:", src_hash)
print("Bronze MD5:  ", bronze_hash)
print("Bronze copy is identical to the provided file.")

Original MD5: 2b7d6737635600be42aa639140cd6ea9
Bronze MD5:   2b7d6737635600be42aa639140cd6ea9
Bronze copy is identical to the provided file.


In [4]:
for root, dirs, fnames in sorted(os.walk(DATA_LAKE)):
    level = root.replace(DATA_LAKE, "").count(os.sep)
    print("    " * level + os.path.basename(root) + "/")
    for name in sorted(fnames):
        print("    " * (level + 1) + name)

data_lake/
    bronze/
        sales.csv
    silver/


## 2. Load and Inspect with Spark

### 2.1 SparkSession and reading from Bronze

Two settings are fixed explicitly so the date handling behaves the same on different PySpark versions: ANSI mode is turned off (so an invalid date becomes `null` instead of failing the job), and the `CORRECTED` time parser policy is used so impossible dates such as 31 February are rejected rather than rolled over.

In [5]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.utils import AnalysisException

spark = (
    SparkSession.builder
    .appName("SalesDataLake")
    .config("spark.sql.ansi.enabled", "false")
    .config("spark.sql.legacy.timeParserPolicy", "CORRECTED")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)

Spark version: 4.0.5


In [6]:
EXPECTED_COLUMNS = [
    "order_id", "customer_id", "customer_name", "product", "category",
    "quantity", "unit_price", "discount_percent", "payment_method",
    "city", "state", "order_date", "order_status",
]

try:
    bronze_df = spark.read.csv(BRONZE_FILE, header=True, inferSchema=True)
except AnalysisException as e:
    raise RuntimeError(f"Could not read Bronze file: {e}")

missing_cols = set(EXPECTED_COLUMNS) - set(bronze_df.columns)
if missing_cols:
    raise ValueError(f"Missing expected columns: {missing_cols}")

bronze_df.cache()
bronze_count = bronze_df.count()
if bronze_count == 0:
    raise ValueError("The Bronze file contains no records.")

### 2.2 Schema, sample rows and record count

In [7]:
bronze_df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)



`order_date` was inferred as a string, which already suggests that not every value is a valid date.

In [8]:
bronze_df.show(5, truncate=False)
print("Total records in Bronze:", bronze_count)

+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name |product      |category|quantity|unit_price|discount_percent|payment_method|city      |state         |order_date|order_status|
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|100001  |C0256      |Keerthi Mishra|Saree        |Clothing|2       |6710      |20              |Credit Card   |Mumbai    |Maharashtra   |2026-01-01|Delivered   |
|100002  |C0042      |Arjun Singh   |T-Shirt      |Clothing|6       |680       |5               |Debit Card    |Chennai   |Tamil Nadu    |2025-08-03|Delivered   |
|100003  |C0348      |Keerthi Rao   |Hoodie       |Clothing|6       |3080      |12              |UPI           |Pune      |Maharashtra   |2025-05-06|Shipped     |
|100004  |C0051      |

### 2.3 Data quality checks

**(a) Duplicates**


In [9]:
distinct_count = bronze_df.distinct().count()
print("Exact duplicate rows:", bronze_count - distinct_count)

(bronze_df.groupBy(bronze_df.columns).count()
    .filter("count > 1")
    .select("order_id", "customer_name", "product", "order_date", "count")
    .orderBy("order_id")
    .show(truncate=False))

Exact duplicate rows: 10
+--------+----------------+-----------------+----------+-----+
|order_id|customer_name   |product          |order_date|count|
+--------+----------------+-----------------+----------+-----+
|100055  |Pranav Iyer     |Dry Fruits Pack  |2026-08-20|2    |
|100145  |Sai Mishra      |Face Wash        |2025-08-28|2    |
|100297  |Manish Pillai   |Dry Fruits Pack  |2025-07-29|2    |
|100317  |Siddharth Mishra|Kurta            |2025-02-04|2    |
|100329  |Varun Verma     |Bedside Table    |2025-04-08|2    |
|100444  |Abhinav Pillai  |Novel            |2025-09-11|2    |
|100502  |Vikram Yadav    |Induction Cooktop|2025-03-11|2    |
|100588  |Diya Tripathi   |Bookshelf        |2026-08-09|2    |
|100692  |Varun Agarwal   |Electric Kettle  |2026-08-18|2    |
|100773  |Meera Malhotra  |Study Table      |2026-02-25|2    |
+--------+----------------+-----------------+----------+-----+



In [10]:
dup_ids = (bronze_df.distinct()
           .groupBy("order_id").count()
           .filter("count > 1"))
print("order_id values shared by different (non-identical) rows:", dup_ids.count())

(bronze_df.distinct()
    .join(dup_ids.select("order_id"), "order_id")
    .select("order_id", "customer_id", "customer_name", "product", "order_date")
    .orderBy("order_id")
    .show(truncate=False))

order_id values shared by different (non-identical) rows: 8
+--------+-----------+------------------+----------------------+----------+
|order_id|customer_id|customer_name     |product               |order_date|
+--------+-----------+------------------+----------------------+----------+
|100064  |C0276      |Ananya Agarwal    |Microwave Oven        |2025-05-15|
|100064  |C0149      |Manish Pillai     |Basmati Rice 5kg      |2026-08-31|
|100072  |C0312      |Aarav Kulkarni    |Face Wash             |2026-08-15|
|100072  |C0211      |Meera Chopra      |Competitive Exam Guide|2026-06-30|
|100305  |C0199      |Pooja Reddy       |Cooking Oil 5L        |2025-06-16|
|100305  |C0076      |Neha Reddy        |Office Chair          |2025-12-29|
|100331  |C0005      |Sakshi Joshi      |Football              |2025-06-06|
|100331  |C0137      |Riya Kapoor       |Jacket                |2025-02-20|
|100434  |C0313      |Sai Das           |Business Book         |2026-03-30|
|100434  |C0149      |Manish

These rows share an `order_id` but have different customers, products and dates, so they look like separate orders with a reused ID rather than true duplicates. They are kept.

**(b) Null / empty values**

In [11]:
string_cols = [c for c, t in bronze_df.dtypes if t == "string"]

null_counts = bronze_df.select([
    F.sum(
        (F.col(c).isNull() | (F.trim(F.col(c)) == "")).cast("int")
        if c in string_cols else F.col(c).isNull().cast("int")
    ).alias(c)
    for c in bronze_df.columns
])
null_counts.show(truncate=False)

+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|0       |0          |10           |0      |0       |0       |0         |0               |8             |8   |0    |0         |0           |
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+



**(c) Text-format inconsistencies**


In [12]:
bronze_df.select([
    F.sum((F.col(c) != F.trim(F.col(c))).cast("int")).alias(c)
    for c in string_cols
]).show(truncate=False)

+-----------+-------------+-------+--------+--------------+----+-----+----------+------------+
|customer_id|customer_name|product|category|payment_method|city|state|order_date|order_status|
+-----------+-------------+-------+--------+--------------+----+-----+----------+------------+
|0          |3            |0      |0       |0             |3   |0    |0         |0           |
+-----------+-------------+-------+--------+--------------+----+-----+----------+------------+



In [13]:
for c in ["category", "city"]:
    print(f"Distinct raw values in '{c}':")
    (bronze_df.groupBy(F.concat(F.lit("'"), F.col(c), F.lit("'")).alias(c))
        .count().orderBy(c).show(30, truncate=False))

Distinct raw values in 'category':
+-----------------+-----+
|category         |count|
+-----------------+-----+
|'Beauty'         |136  |
|'Books'          |128  |
|'CLOTHING'       |1    |
|'Clothing'       |108  |
|'ELECTRONICS'    |1    |
|'Electronics'    |103  |
|'Furniture'      |132  |
|'Grocery'        |136  |
|'Home Appliances'|115  |
|'SPORTS'         |1    |
|'Sports'         |136  |
|'beauty'         |2    |
|'sports'         |1    |
+-----------------+-----+

Distinct raw values in 'city':
+-------------+-----+
|city         |count|
+-------------+-----+
|NULL         |8    |
|' Hyderabad '|2    |
|' Mumbai '   |1    |
|'Bengaluru'  |91   |
|'Chennai'    |88   |
|'Coimbatore' |81   |
|'DELHI'      |2    |
|'Delhi'      |81   |
|'HYDERABAD'  |1    |
|'Hyderabad'  |94   |
|'JAIPUR'     |1    |
|'Jaipur'     |84   |
|'Kolkata'    |81   |
|'Lucknow'    |90   |
|'Mumbai'     |106  |
|'Pune'       |82   |
|'Vijayawada' |105  |
|'coimbatore' |1    |
|'delhi'      |1    |
+------

**(d) Invalid numerical values**

Rules used: `quantity` must be at least 1, `unit_price` must be greater than 0, and `discount_percent` must be between 0 and 100.

In [14]:
num_cols = ["quantity", "unit_price", "discount_percent"]
bronze_df.select(num_cols).summary("count", "min", "max", "mean").show()

invalid_rules = {
    "quantity": F.col("quantity") < 1,
    "unit_price": F.col("unit_price") <= 0,
    "discount_percent": (F.col("discount_percent") < 0) | (F.col("discount_percent") > 100),
}
for name, cond in invalid_rules.items():
    bad = bronze_df.filter(cond)
    print(f"{name}: {bad.count()} invalid value(s) ->",
          sorted({r[0] for r in bad.select(name).collect()}))

+-------+--------+----------+----------------+
|summary|quantity|unit_price|discount_percent|
+-------+--------+----------+----------------+
|  count|    1000|      1000|            1000|
|    min|      -2|      -500|              -5|
|    max|      10|     81010|             120|
|   mean|   5.355|    5702.4|          19.948|
+-------+--------+----------+----------------+

quantity: 12 invalid value(s) -> [-2, -1, 0]
unit_price: 8 invalid value(s) -> [-500, -250, 0]
discount_percent: 8 invalid value(s) -> [-5, 120]


**(e) Inconsistent / invalid dates**

The expected format is `yyyy-MM-dd`. Values that do not match the pattern, or match it but are not real calendar dates, are listed below.

In [15]:
iso_pattern = r"^\d{4}-\d{2}-\d{2}$"

bad_dates = bronze_df.filter(
    ~F.col("order_date").rlike(iso_pattern)
    | F.to_date("order_date", "yyyy-MM-dd").isNull()
)
print("Rows with a non-standard or invalid order_date:", bad_dates.count())
bad_dates.select("order_id", "order_date").show(truncate=False)

Rows with a non-standard or invalid order_date: 6
+--------+----------+
|order_id|order_date|
+--------+----------+
|100097  |31/02/2026|
|100358  |2025/07/14|
|100448  |not-a-date|
|100634  |2025-02-30|
|100637  |2026-13-05|
|100969  |09-15-2026|
+--------+----------+



Summary of what the inspection found:

- 10 exact duplicate rows, plus a few reused `order_id` values on otherwise different orders.
- Blank values in `customer_name`, `payment_method` and `city`.
- Extra spaces and mixed capitalisation in `customer_name`, `category` and `city`.
- Zero or negative `quantity`, zero or negative `unit_price`, and discounts below 0% or above 100%.
- A handful of dates in other formats (`yyyy/MM/dd`, `dd/MM/yyyy`, `MM-dd-yyyy`) and some that are impossible (month 13 and 31 February) or not dates at all.

## 3. Basic Preprocessing - Silver Layer

### Step 1: Remove exact duplicate rows

In [16]:
step1 = bronze_df.dropDuplicates()
print("Rows before:", bronze_count, "| after removing exact duplicates:", step1.count())

Rows before: 1000 | after removing exact duplicates: 990


### Step 2 and 3: Clean text fields and handle missing values

Trimming is done before filling missing values so that strings made only of spaces are also treated as missing.

- All text columns are trimmed, and empty strings are converted to `null`.
- `customer_name`, `category`, `city` and `state` are converted to title case.
- A missing `customer_name` is filled from another order of the same `customer_id` where one exists, otherwise `"Unknown"`.
- A missing `payment_method` or `city` is filled with `"Unknown"`.

In [17]:
TITLE_CASE_COLS = ["customer_name", "category", "city", "state"]

step2 = step1
for c in string_cols:
    cleaned = F.trim(F.col(c))
    if c in TITLE_CASE_COLS:
        cleaned = F.initcap(F.lower(cleaned))
    step2 = step2.withColumn(c, F.when(cleaned == "", None).otherwise(cleaned))

name_lookup = (step2.filter(F.col("customer_name").isNotNull())
               .groupBy("customer_id")
               .agg(F.first("customer_name", ignorenulls=True).alias("known_name")))

step3 = (step2.join(name_lookup, "customer_id", "left")
         .withColumn("customer_name",
                     F.coalesce("customer_name", "known_name", F.lit("Unknown")))
         .drop("known_name")
         .fillna({"payment_method": "Unknown", "city": "Unknown"})
         .select(EXPECTED_COLUMNS))

step3.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in string_cols]).show()
step3.groupBy("category").count().orderBy("category").show(truncate=False)

+-----------+-------------+-------+--------+--------------+----+-----+----------+------------+
|customer_id|customer_name|product|category|payment_method|city|state|order_date|order_status|
+-----------+-------------+-------+--------+--------------+----+-----+----------+------------+
|          0|            0|      0|       0|             0|   0|    0|         0|           0|
+-----------+-------------+-------+--------+--------------+----+-----+----------+------------+

+---------------+-----+
|category       |count|
+---------------+-----+
|Beauty         |137  |
|Books          |127  |
|Clothing       |108  |
|Electronics    |104  |
|Furniture      |129  |
|Grocery        |134  |
|Home Appliances|113  |
|Sports         |138  |
+---------------+-----+



### Step 4: Validate quantity, unit price and discount

Values are cast safely (anything non-numeric becomes `null`). Clearly invalid values are set to `null` instead of dropping the whole row, so the rest of the order information is not lost.

In [18]:
step4 = (step3
    .withColumn("quantity", F.expr("try_cast(quantity AS INT)"))
    .withColumn("unit_price", F.expr("try_cast(unit_price AS DOUBLE)"))
    .withColumn("discount_percent", F.expr("try_cast(discount_percent AS DOUBLE)"))
    .withColumn("quantity", F.when(F.col("quantity") >= 1, F.col("quantity")))
    .withColumn("unit_price", F.when(F.col("unit_price") > 0, F.col("unit_price")))
    .withColumn("discount_percent",
                F.when(F.col("discount_percent").between(0, 100), F.col("discount_percent"))))

step4.select([F.sum(F.col(c).isNull().cast("int")).alias(f"{c}_set_null")
              for c in num_cols]).show()

+-----------------+-------------------+-------------------------+
|quantity_set_null|unit_price_set_null|discount_percent_set_null|
+-----------------+-------------------+-------------------------+
|               12|                  8|                        8|
+-----------------+-------------------+-------------------------+



### Step 5: Convert dates to one format

Each date is matched against the formats that actually appear in the data and parsed with the matching pattern. Anything that cannot be parsed into a real date becomes `null`. The result is a proper `DateType` column, written as `yyyy-MM-dd`.

In [19]:
d = F.col("order_date")
parsed_date = (
    F.when(d.rlike(r"^\d{4}-\d{2}-\d{2}$"), F.to_date(d, "yyyy-MM-dd"))
     .when(d.rlike(r"^\d{4}/\d{2}/\d{2}$"), F.to_date(d, "yyyy/MM/dd"))
     .when(d.rlike(r"^\d{2}/\d{2}/\d{4}$"), F.to_date(d, "dd/MM/yyyy"))
     .when(d.rlike(r"^\d{2}-\d{2}-\d{4}$"), F.to_date(d, "MM-dd-yyyy"))
)

step5 = step4.withColumn("order_date_clean", parsed_date)

(step5.filter(~d.rlike(r"^\d{4}-\d{2}-\d{2}$") | F.col("order_date_clean").isNull())
    .select("order_id", "order_date", "order_date_clean")
    .show(truncate=False))

step5 = step5.withColumn("order_date", F.col("order_date_clean")).drop("order_date_clean")

+--------+----------+----------------+
|order_id|order_date|order_date_clean|
+--------+----------+----------------+
|100097  |31/02/2026|NULL            |
|100637  |2026-13-05|NULL            |
|100969  |09-15-2026|2026-09-15      |
|100448  |not-a-date|NULL            |
|100634  |2025-02-30|NULL            |
|100358  |2025/07/14|2025-07-14      |
+--------+----------+----------------+



### Step 6: Preserving records

No rows are dropped apart from the exact duplicates. Rows with an invalid number or date keep all their other fields and only the bad value is `null`. The reused `order_id` rows are also kept. A final type cast gives the Silver table a clean schema.

In [20]:
silver_df = (step5
    .withColumn("order_id", F.col("order_id").cast("int"))
    .select(EXPECTED_COLUMNS))

silver_df.printSchema()
silver_count = silver_df.count()
print("Rows going to Silver:", silver_count)

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = false)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_percent: double (nullable = true)
 |-- payment_method: string (nullable = false)
 |-- city: string (nullable = false)
 |-- state: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = true)

Rows going to Silver: 990


### Saving to the Silver layer (Parquet)

In [21]:
try:
    silver_df.write.mode("overwrite").parquet(SILVER_PATH)
except Exception as e:
    raise RuntimeError(f"Failed to write Silver data: {e}")

print("Silver data written to:", SILVER_PATH)
print(sorted(f for f in os.listdir(SILVER_PATH) if not f.startswith(".")))

Silver data written to: /content/data_lake/silver/preprocessed_sales
['_SUCCESS', 'part-00000-825e047d-1b62-484e-b356-f345315fe4b7-c000.snappy.parquet']


###Export Silver Dataset to CSV Format

In [22]:
import glob

CSV_FILE = os.path.join(SILVER_DIR, "sales_silver.csv")
TMP_DIR = os.path.join(SILVER_DIR, "_tmp_csv")

if not os.path.isdir(SILVER_PATH):
    raise FileNotFoundError(f"Parquet folder not found: {SILVER_PATH}")

silver_parquet = spark.read.parquet(SILVER_PATH)

# Spark always writes a folder of part files, so write to a temp folder
# and then move the single part file to the required name
(silver_parquet.orderBy("order_id").coalesce(1)
    .write.mode("overwrite")
    .option("header", True)
    .option("dateFormat", "yyyy-MM-dd")
    .csv(TMP_DIR))

part_files = glob.glob(os.path.join(TMP_DIR, "part-*.csv"))
if len(part_files) != 1:
    raise RuntimeError(f"Expected one CSV part file, found {len(part_files)}")

shutil.move(part_files[0], CSV_FILE)
shutil.rmtree(TMP_DIR)

check = spark.read.csv(CSV_FILE, header=True, schema=silver_parquet.schema)
print("Saved:", CSV_FILE)
print("Rows in Parquet:", silver_parquet.count(), "| Rows in CSV:", check.count())
check.show(5, truncate=False)

Saved: /content/data_lake/silver/sales_silver.csv
Rows in Parquet: 990 | Rows in CSV: 990
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name |product      |category|quantity|unit_price|discount_percent|payment_method|city      |state         |order_date|order_status|
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|100001  |C0256      |Keerthi Mishra|Saree        |Clothing|2       |6710.0    |20.0            |Credit Card   |Mumbai    |Maharashtra   |2026-01-01|Delivered   |
|100002  |C0042      |Arjun Singh   |T-Shirt      |Clothing|6       |680.0     |5.0             |Debit Card    |Chennai   |Tamil Nadu    |2025-08-03|Delivered   |
|100003  |C0348      |Keerthi Rao   |Hoodie       |Clothing|6       |3080.0    |12.0           

## 4. Verification

### 4.1 Row count before and after preprocessing

In [23]:
silver_read = spark.read.parquet(SILVER_PATH)
silver_read_count = silver_read.count()

print(f"Bronze (before preprocessing): {bronze_count}")
print(f"Silver (after preprocessing):  {silver_read_count}")
print(f"Rows removed (exact duplicates): {bronze_count - silver_read_count}")
assert silver_read_count == silver_count, "Row count changed after writing to Silver."

Bronze (before preprocessing): 1000
Silver (after preprocessing):  990
Rows removed (exact duplicates): 10


### 4.2 Reading the Silver data back with Spark

In [24]:
silver_read.printSchema()
silver_read.orderBy("order_id").show(10, truncate=False)

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_percent: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = true)

+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name  |product              |category |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+---------------+---------------------+---------+--------+----------+------------

In [25]:
print("Remaining exact duplicates:", silver_read_count - silver_read.distinct().count())
print("Text values with extra spaces:",
      silver_read.filter(" OR ".join(f"{c} != trim({c})" for c in string_cols)).count())
silver_read.groupBy("city").count().orderBy("city").show(truncate=False)

Remaining exact duplicates: 0
Text values with extra spaces: 0
+----------+-----+
|city      |count|
+----------+-----+
|Bengaluru |90   |
|Chennai   |88   |
|Coimbatore|81   |
|Delhi     |83   |
|Hyderabad |97   |
|Jaipur    |83   |
|Kolkata   |81   |
|Lucknow   |89   |
|Mumbai    |105  |
|Pune      |81   |
|Unknown   |8    |
|Vijayawada|104  |
+----------+-----+



In [26]:
spark.catalog.clearCache()
for root, dirs, fnames in sorted(os.walk(DATA_LAKE)):
    level = root.replace(DATA_LAKE, "").count(os.sep)
    print("    " * level + os.path.basename(root) + "/")
    for name in sorted(f for f in fnames if not f.startswith(".")):
        print("    " * (level + 1) + name)

data_lake/
    bronze/
        sales.csv
    silver/
        sales_silver.csv
        preprocessed_sales/
            _SUCCESS
            part-00000-825e047d-1b62-484e-b356-f345315fe4b7-c000.snappy.parquet


### 4.3 Preprocessing operations performed

1. **Duplicates:** removed exact duplicate rows with `dropDuplicates()`. Rows sharing an `order_id` but with different details were kept.
2. **Missing values:** blank strings were converted to `null`. Missing customer names were filled from the same `customer_id` where possible, otherwise `"Unknown"`. Missing `payment_method` and `city` were set to `"Unknown"`.
3. **Text cleanup:** all text fields were trimmed. `customer_name`, `category`, `city` and `state` were converted to title case (for example `DELHI` and `delhi` became `Delhi`).
4. **Numeric validation:** `quantity` below 1, `unit_price` of 0 or less, and `discount_percent` outside 0 to 100 were set to `null`, and the columns were cast to numeric types.
5. **Dates:** `yyyy-MM-dd`, `yyyy/MM/dd`, `dd/MM/yyyy` and `MM-dd-yyyy` values were all converted to a single `DateType` column. Impossible or unreadable dates were set to `null`.
6. **Record preservation:** apart from duplicates, no rows were removed. Only the specific invalid values were nulled.

The Bronze file was left untouched (checked with MD5), and the cleaned data is stored as Parquet in `data_lake/silver/preprocessed_sales/`.

In [27]:
spark.stop()